# UJIAN TENGAH SEMESTER (UTS) TEMU KEMBALI INFORMASI
## Tugas 3: Web Crawler & Information Extraction Dokumen Putusan Mahkamah Agung RI

---
**Identitas Mahasiswa:**
- **Nama Mahasiswa** : Adnan Amhar
- **NIM** : 202310370311001 (3 Digit Terakhir: 001)
- **Mata Kuliah** : Temu Kembali Informasi
- **Dosen Pengampu** : Galih W. W
- **Institusi** : Fakultas Teknik, Universitas Muhammadiyah Malang

**Parameter Target:**
- **Lembaga Peradilan** : Pengadilan Negeri Jakarta Timur (`PN Jakarta Timur`)
- **Klasifikasi Perkara** : Pidana Khusus -> Narkotika dan Psikotropika
- **Tahun Putusan** : 2023 - 2025
- **Jumlah Korpus** : 50 Dokumen Putusan
- **Format Ekstraksi** : 9 Kolom Terstruktur (5 Kolom Wajib + 4 Kolom Bonus Nilai Plus)
---

### 1. Instalasi Pustaka Pendukung
Menginstal modul yang dibutuhkan untuk crawling, pembacaan berkas PDF (`PyMuPDF` dan `pypdf`), manipulasi data (`pandas`), dan visualisasi progress.

In [ ]:
!pip install -q requests beautifulsoup4 pymupdf pypdf pandas tqdm

### 2. Import Modul & Konfigurasi Path
Menyiapkan modul Python standar dan struktur folder penyimpanan untuk menampung file PDF serta teks (.txt) hasil crawling.

In [ ]:
import os
import re
import time
from pathlib import Path
import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm import tqdm
import fitz  # PyMuPDF

# Direktori penyimpanan
DIR_PDF = Path("data_putusan_55/pdf") if Path("data_putusan_55/pdf").exists() else Path("data_putusan_50/pdf")
DIR_TXT = Path("data_putusan_55/txt") if Path("data_putusan_55/txt").exists() else Path("data_putusan_50/txt")
DIR_PDF.mkdir(parents=True, exist_ok=True)
DIR_TXT.mkdir(parents=True, exist_ok=True)

print("Folder kerja berhasil disiapkan.")

### 3. Model Web Crawler Santun (Polite Crawler)
Sesuai indikator CPMK-3 (memahami kebijakan `robots.txt` dan etika crawling), scraper dilengkapi dengan header `User-Agent` yang wajar, jeda waktu request berkala, dan mekanisme penanganan error.

In [ ]:
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'id-ID,id;q=0.9,en-US;q=0.8,en;q=0.7'
}

BASE_MA_URL = "https://putusan3.mahkamahagung.go.id/direktori/kategori/jenis/pidana-khusus/klasifikasi/narkotika-dan-psikotropika.html"

def unduh_file_pdf(url_download, target_path):
    """Mengunduh berkas PDF putusan dengan penanganan timeout."""
    try:
        resp = requests.get(url_download, headers=HEADERS, timeout=25, stream=True)
        if resp.status_code == 200:
            with open(target_path, 'wb') as f:
                for chunk in resp.iter_content(chunk_size=8192):
                    f.write(chunk)
            return True
    except Exception as e:
        print(f"Gagal mengunduh {url_download}: {e}")
    return False

### 4. Batch Converter PDF ke Teks Murni (.txt)
Mengekstrak seluruh halaman naskah putusan pengadilan menjadi file teks murni (.txt) menggunakan `PyMuPDF`.

In [ ]:
def konversi_pdf_ke_txt(pdf_file, txt_file):
    """Ekstraksi teks utuh dari berkas PDF ke file .txt."""
    try:
        doc = fitz.open(pdf_file)
        teks = "\n".join([page.get_text() for page in doc])
        doc.close()
        with open(txt_file, 'w', encoding='utf-8') as f:
            f.write(teks)
        return teks
    except Exception as e:
        print(f"Error membaca {pdf_file}: {e}")
        return ""

### 5. Mesin Ekstraksi Informasi Berbasis Regex (Information Extraction)
Mengekstrak **9 kolom terstruktur**:
- **5 Kolom Wajib**: `No`, `No Putusan`, `Lembaga Peradilan`, `Barang Bukti`, `Amar Putusan`.
- **4 Kolom Nilai Plus**: `Nama Terdakwa`, `Tanggal Putusan`, `Lama Pidana Penjara`, `Pidana Denda & Subsider`.

In [ ]:
def ekstrak_fitur_putusan(no_urut, teks, nama_file=""):
    """Mengekstrak 9 atribut terstruktur dari teks putusan pengadilan."""
    # 1. No Putusan
    m_no = re.search(r'(?:Nomor|NOMOR)\s*:?\s*([0-9]+/[Pp]id\.[A-Za-z0-9\.]+(?:/[A-Za-z0-9\.]+)?/[0-9]{4}/PN\s*[A-Za-z\.]+)', teks)
    no_putusan = m_no.group(1).strip() if m_no else nama_file.replace('.txt','').replace('.pdf','')
    
    # 2. Amar Putusan
    pola_amar = r'(?:M\s*E\s*N\s*G\s*A\s*D\s*I\s*L\s*I\s*:?|MENGADILI\s*:?)(.*?)(?:Demikianlah|Demikian\s+diputuskan|Panitera\s+Pengganti|$)'
    m_amar = re.search(pola_amar, teks, re.DOTALL | re.IGNORECASE)
    amar = m_amar.group(1).strip() if m_amar else "Amar putusan tidak dapat diekstraksi"
    amar_clean = re.sub(r'\s{2,}', ' ', amar)
    
    # 3. Barang Bukti
    pola_bb = r'Menetapkan\s+barang\s+bukti\s+berupa\s*:?\s*(.*?)(?:Dirampas|Dikembalikan|Dimusnahkan|Membebankan|Menetapkan\s+agar|6\.)'
    m_bb = re.search(pola_bb, teks, re.DOTALL | re.IGNORECASE)
    barang_bukti = re.sub(r'\s{2,}', ' ', m_bb.group(1).strip(' -:;')) if m_bb else "Barang bukti tidak terperinci"
    
    # 4. Nama Terdakwa
    m_nama = re.search(r'Nama\s+lengkap\s*:?\s*([^\n\r;]+)', teks, re.IGNORECASE)
    if m_nama:
        nama_terdakwa = m_nama.group(1).strip()
    else:
        m_nama_amar = re.search(r'Menyatakan\s+Terdakwa\s+([A-Z\s\.\',]+?)(?:\s+tersebut|\s+telah|\s+terbukti|,)', teks)
        nama_terdakwa = m_nama_amar.group(1).strip() if m_nama_amar else "Tidak Teridentifikasi"
        
    # 5. Tanggal Putusan
    bulan_regex = r'(?:Januari|Februari|Maret|April|Mei|Juni|Juli|Agustus|September|Oktober|November|Desember)'
    m_tgl = re.search(rf'tanggal\s+([0-9]{{1,2}}\s+{bulan_regex}\s+[0-9]{{4}})', teks, re.IGNORECASE)
    tgl_putusan = m_tgl.group(1).strip() if m_tgl else "Tidak Tertera"
    
    # 6. Lama Pidana Penjara
    m_penjara = re.search(r'penjara\s+selama\s+([0-9]+\s*(?:\([^\)]+\))?\s*(?:tahun|bulan)(?:\s+(?:dan\s+)?[0-9]+\s*(?:\([^\)]+\))?\s*bulan)?)', amar_clean, re.IGNORECASE)
    lama_penjara = m_penjara.group(1).strip() if m_penjara else "Tidak Ditemukan"
    
    # 7. Pidana Denda & Subsider
    m_denda = re.search(r'(denda\s+(?:sejumlah|sebesar)\s+Rp\.?\s*[0-9\.,]+[^\.;]+(?:dengan\s+ketentuan|apabila|jika|subsider|diganti)[^\.;]+)', amar_clean, re.IGNORECASE)
    denda = re.sub(r'\s{2,}', ' ', m_denda.group(1).strip()) if m_denda else "Nihil / Tidak Dikenakan Denda"
    
    return {
        "No": no_urut,
        "No Putusan": no_putusan,
        "Lembaga Peradilan": "PN Jakarta Timur",
        "Barang Bukti": barang_bukti,
        "Amar Putusan": amar_clean,
        "Nama Terdakwa": nama_terdakwa,
        "Tanggal Putusan": tgl_putusan,
        "Lama Pidana Penjara": lama_penjara,
        "Pidana Denda & Subsider": denda
    }

### 6. Eksekusi Ekstraksi Batch & Pratinjau DataFrame
Memproses seluruh berkas PDF putusan yang telah diunduh, menyimpan file teksnya ke folder `data_putusan_50/txt/`, dan menyusun tabel hasil ekstraksi ke format CSV.

In [ ]:
pdf_list = sorted(list(DIR_PDF.glob("*.pdf")))
print(f"Menemukan {len(pdf_list)} file PDF untuk diekstrak.")

hasil_data = []
for idx, p_file in enumerate(pdf_list, start=1):
    txt_file = DIR_TXT / f"{p_file.stem}.txt"
    teks = konversi_pdf_ke_txt(str(p_file), str(txt_file))
    fitur = ekstrak_fitur_putusan(no_urut=idx, teks=teks, nama_file=p_file.name)
    hasil_data.append(fitur)

df_hasil = pd.DataFrame(hasil_data)
csv_output = "hasil_ekstraksi_50_putusan.csv"
df_hasil.to_csv(csv_output, index=False, encoding='utf-8-sig')
print(f"\nEkstraksi selesai! File tersimpan di: {csv_output}")

# Tampilkan 5 baris pertama
df_hasil.head()

### 7. Pengarsipan untuk Pengumpulan Google Drive
Mengompres folder `pdf` dan `txt` menjadi file `.zip` agar siap diunggah ke Google Drive dan dibagikan tautan publiknya.

In [ ]:
!zip -r data_putusan_55_pdf_txt.zip data_putusan_55/ hasil_ekstraksi_50_putusan.csv 2>/dev/null || zip -r data_putusan_50_pdf_txt.zip data_putusan_50/ hasil_ekstraksi_50_putusan.csv
print("Arsip data_putusan_50_pdf_txt.zip siap diunggah ke Google Drive.")